# Task 10: Wasserstein GAN with Gradient Penalty (WGAN-GP)

## Objective

Implement a Wasserstein Generative Adversarial Network with Gradient Penalty (WGAN-GP) for stable image synthesis.

The implementation includes:

- Generator architecture
- Critic architecture
- Wasserstein critic loss
- Generator loss
- Interpolated samples
- Gradient penalty
- 1-Lipschitz constraint
- 100-epoch training
- Training-loss tracking
- Generated image visualization

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

from torch.utils.data import DataLoader, Subset

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Device:", device)


# ============================================================
# MNIST Dataset
# ============================================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Smaller subset keeps 100 epochs practical in Colab
dataset = Subset(
    dataset,
    range(10000)
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    drop_last=True
)

print("Training samples:", len(dataset))

# Generator and Critic

The generator transforms random noise:

\[
z\sim N(0,I)
\]

into a synthetic \(28\times28\) image.

The critic receives an image and outputs a single unrestricted scalar rather than a probability.

This is an important difference from the original GAN discriminator.

The critic does **not** use a final sigmoid layer.

In [ ]:
class Generator(nn.Module):

    def __init__(self, latent_dim=64):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(
                latent_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                512
            ),

            nn.ReLU(),

            nn.Linear(
                512,
                1024
            ),

            nn.ReLU(),

            nn.Linear(
                1024,
                28 * 28
            ),

            nn.Tanh()
        )

    def forward(self, z):

        x = self.model(z)

        return x.view(
            -1,
            1,
            28,
            28
        )


class Critic(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(
                28 * 28,
                1024
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Linear(
                1024,
                512
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Linear(
                512,
                256
            ),

            nn.LeakyReLU(
                0.2
            ),

            # No sigmoid
            nn.Linear(
                256,
                1
            )
        )

    def forward(self, x):

        x = x.view(
            x.size(0),
            -1
        )

        return self.model(x)


LATENT_DIM = 64

G = Generator(
    LATENT_DIM
).to(device)

D = Critic().to(device)

print(
    "Generator parameters:",
    sum(p.numel() for p in G.parameters())
)

print(
    "Critic parameters:",
    sum(p.numel() for p in D.parameters())
)

# Gradient Penalty

The gradient penalty is calculated on interpolated samples between real and generated images.

For:

\[
\hat{x}=
\epsilon x+(1-\epsilon)x_{fake}
\]

we calculate:

\[
\nabla_{\hat{x}}D(\hat{x})
\]

and penalize deviations from unit gradient norm:

\[
L_{GP}
=
\left(
\|\nabla_{\hat{x}}D(\hat{x})\|_2-1
\right)^2
\]

In [ ]:
def gradient_penalty(
    critic,
    real,
    fake
):

    batch_size = real.size(0)

    # Random interpolation coefficient
    epsilon = torch.rand(
        batch_size,
        1,
        1,
        1,
        device=device
    )

    # Interpolated samples
    interpolated = (
        epsilon * real
        +
        (1 - epsilon) * fake
    )

    interpolated.requires_grad_(True)

    critic_output = critic(
        interpolated
    )

    # Gradient of critic output
    # with respect to interpolated input
    gradients = torch.autograd.grad(
        outputs=critic_output,
        inputs=interpolated,
        grad_outputs=torch.ones_like(
            critic_output
        ),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(
        batch_size,
        -1
    )

    gradient_norm = torch.sqrt(
        torch.sum(
            gradients ** 2,
            dim=1
        ) + 1e-12
    )

    penalty = torch.mean(
        (gradient_norm - 1) ** 2
    )

    return penalty, gradient_norm.mean()

# WGAN-GP Training

The critic is updated several times for every generator update.

### Critic

\[
L_D =
E[D(fake)]
-
E[D(real)]
+
\lambda L_{GP}
\]

### Generator

\[
L_G=-E[D(fake)]
\]

The training is performed for 100 epochs while storing:

- Critic loss
- Generator loss
- Gradient penalty
- Gradient norm

These values are later used to analyze training stability.

In [ ]:
# ============================================================
# Training configuration
# ============================================================

EPOCHS = 100

LR = 1e-4

BETA1 = 0.0
BETA2 = 0.9

LAMBDA_GP = 10

CRITIC_STEPS = 3


optimizer_G = optim.Adam(
    G.parameters(),
    lr=LR,
    betas=(BETA1, BETA2)
)

optimizer_D = optim.Adam(
    D.parameters(),
    lr=LR,
    betas=(BETA1, BETA2)
)


critic_history = []
generator_history = []
gp_history = []
gradient_history = []


# Fixed noise for visual comparison
fixed_noise = torch.randn(
    16,
    LATENT_DIM,
    device=device
)


# ============================================================
# Training
# ============================================================

for epoch in range(EPOCHS):

    G.train()
    D.train()

    epoch_D = 0
    epoch_G = 0
    epoch_GP = 0
    epoch_grad = 0

    for real, _ in loader:

        real = real.to(device)

        # ----------------------------------------------------
        # Train Critic
        # ----------------------------------------------------

        for _ in range(CRITIC_STEPS):

            optimizer_D.zero_grad()

            z = torch.randn(
                real.size(0),
                LATENT_DIM,
                device=device
            )

            fake = G(z).detach()

            real_score = D(real)
            fake_score = D(fake)

            gp, grad_norm = gradient_penalty(
                D,
                real,
                fake
            )

            critic_loss = (
                fake_score.mean()
                - real_score.mean()
                + LAMBDA_GP * gp
            )

            critic_loss.backward()

            optimizer_D.step()

        # ----------------------------------------------------
        # Train Generator
        # ----------------------------------------------------

        optimizer_G.zero_grad()

        z = torch.randn(
            real.size(0),
            LATENT_DIM,
            device=device
        )

        fake = G(z)

        generator_loss = -D(
            fake
        ).mean()

        generator_loss.backward()

        optimizer_G.step()

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        epoch_D += critic_loss.item()
        epoch_G += generator_loss.item()
        epoch_GP += gp.item()
        epoch_grad += grad_norm.item()

    batches = len(loader)

    critic_history.append(
        epoch_D / batches
    )

    generator_history.append(
        epoch_G / batches
    )

    gp_history.append(
        epoch_GP / batches
    )

    gradient_history.append(
        epoch_grad / batches
    )

    if (
        epoch == 0
        or (epoch + 1) % 10 == 0
    ):

        print(
            f"Epoch [{epoch+1:3d}/{EPOCHS}] | "
            f"D: {critic_history[-1]:7.3f} | "
            f"G: {generator_history[-1]:7.3f} | "
            f"GP: {gp_history[-1]:6.3f} | "
            f"|grad|: {gradient_history[-1]:.3f}"
        )

In [ ]:
# ============================================================
# Training log analysis
# ============================================================

epochs = np.arange(
    1,
    EPOCHS + 1
)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    critic_history,
    label="Critic Loss"
)

plt.plot(
    epochs,
    generator_history,
    label="Generator Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("WGAN-GP Training Losses")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Gradient penalty and gradient norm
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    gp_history,
    label="Gradient Penalty"
)

plt.plot(
    epochs,
    gradient_history,
    label="Mean Gradient Norm"
)

plt.axhline(
    1.0,
    linestyle="--",
    label="Target Gradient Norm = 1"
)

plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title(
    "WGAN-GP Gradient Constraint Analysis"
)

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# Generate synthetic images
# ============================================================

G.eval()

with torch.no_grad():

    generated = G(
        fixed_noise
    ).cpu()

# Convert [-1, 1] -> [0, 1]
generated = (
    generated + 1
) / 2


plt.figure(
    figsize=(8, 8)
)

for i in range(16):

    plt.subplot(4, 4, i + 1)

    plt.imshow(
        generated[i, 0],
        cmap="gray"
    )

    plt.axis("off")

plt.suptitle(
    "WGAN-GP Generated MNIST Images"
)

plt.tight_layout()
plt.show()


# ============================================================
# Final training statistics
# ============================================================

print("Final Training Statistics")
print("-" * 35)

print(
    f"Final Critic Loss      : "
    f"{critic_history[-1]:.4f}"
)

print(
    f"Final Generator Loss   : "
    f"{generator_history[-1]:.4f}"
)

print(
    f"Final Gradient Penalty : "
    f"{gp_history[-1]:.4f}"
)

print(
    f"Final Gradient Norm    : "
    f"{gradient_history[-1]:.4f}"
)

# Conclusion

A Wasserstein GAN with Gradient Penalty (WGAN-GP) was successfully implemented using PyTorch, TorchVision, and Matplotlib.

The experiment implemented:

- A custom Generator
- A custom Critic
- Wasserstein critic objective
- Generator objective
- Interpolated real/fake samples
- Custom gradient penalty
- Approximate 1-Lipschitz constraint
- 100-epoch training
- Training-log analysis
- Gradient-norm analysis
- Synthetic image generation

The critic does not use a sigmoid output. Instead, it produces an unrestricted scalar score, which is required by the Wasserstein formulation.

The gradient penalty:

\[
L_{GP}
=
E
\left[
\left(
\|\nabla_{\hat{x}}D(\hat{x})\|_2-1
\right)^2
\right]
\]

encourages the gradient norm to remain close to 1 and improves training stability.

The training curves allow the behavior of the Generator, Critic, and gradient penalty to be analyzed over all 100 epochs. The generated samples provide a qualitative evaluation of the learned image distribution.

Overall, the implementation demonstrates how WGAN-GP improves the stability of adversarial training by replacing the traditional discriminator objective with a Wasserstein critic and enforcing the Lipschitz constraint through gradient penalty.